# StreamingT2V Server for the Digital Scout Camp

هذا الـ Notebook يشغّل سيرفر **StreamingT2V** على Google Colab ويفتح رابطًا عامًا عبر **ngrok** يستهلكه تطبيق المخيم الرقمي (لوحة الأدمن وصفحة "تصميم الفيديو").

**خطوات التشغيل (مرة واحدة):**
1. اختر `Runtime → Change runtime type` ثم اختر **GPU** (يفضّل T4 أو أعلى).
2. أنشئ حسابًا مجانيًا على [ngrok](https://dashboard.ngrok.com) واحصل على `Authtoken` من صفحة **Your Authtoken**.
3. أضِف الـ Authtoken إلى Secrets في Colab باسم `NGROK_AUTHTOKEN` (`Tools → Secrets → Add new secret` ثم فعّل خانة *Notebook access*).
4. (اختياري) أضِف Secret آخر باسم `SCOUT_API_TOKEN` بأي نص سرّي تختاره؛ هذا التوكن سيُطلب من التطبيق قبل أي توليد لمنع إساءة الاستخدام.
5. اضغط `Runtime → Run all`. بعد دقائق ستظهر رسالة:
   ```
   >>> Public URL: https://<random>.ngrok-free.app
   ```
6. ادخل إلى **لوحة الأدمن → الإعدادات → خادم StreamingT2V** والصق:
   - `رابط الـ API` = الرابط العام (`https://...ngrok-free.app`).
   - `توكن الحماية` = نفس قيمة `SCOUT_API_TOKEN` (اتركه فارغًا إذا لم تستخدمه).
7. خلّ هذا التبويب مفتوحًا — Colab المجاني يفصل بعد ساعات من الخمول.

كلّ توليد فيديو يستهلك حوالي **2–10 دقائق** على T4 (حسب عدد الإطارات وحجم النموذج). الفيديو يُحفظ داخل ذاكرة Colab المؤقتة فقط.

## 1) فحص الـ GPU

In [ ]:
!nvidia-smi || echo '⚠️ لا يوجد GPU. ارجع وغيّر Runtime Type إلى GPU.'

## 2) استنساخ مستودع StreamingT2V وتثبيت المتطلبات

نستخدم فرع `StreamingModelscope` لأنه أخفّ على الذاكرة من فرع `main` ويناسب وحدات GPU المجانية على Colab.

In [ ]:
%cd /content
import os, pathlib
REPO_DIR = pathlib.Path('/content/StreamingT2V')
if not REPO_DIR.exists():
    !git clone --branch StreamingModelscope --depth 1 https://github.com/Picsart-AI-Research/StreamingT2V.git
%cd /content/StreamingT2V
!python -m pip install --quiet --upgrade pip
# المتطلبات الأصلية تثبّت الموديل + dependencies الثقيلة (torch, xformers...)
!python -m pip install --quiet -r requirements.txt || echo '⚠️ بعض الحزم فشلت — تابع التثبيت يدويًا إن لزم الأمر.'
# حزم خادم الـ API
!python -m pip install --quiet 'fastapi>=0.103' 'uvicorn[standard]>=0.24' pyngrok python-multipart

## 3) تنزيل أوزان النموذج (~ 2 GB)

الـ checkpoint يأتي من [مستودع Hugging Face الرسمي](https://huggingface.co/PAIR/StreamingT2V).

In [ ]:
import pathlib, urllib.request, sys
ckpt_dir = pathlib.Path('/content/StreamingT2V/t2v_enhanced/checkpoints')
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = ckpt_dir / 'streaming_t2v.ckpt'
if not ckpt_path.exists():
    print('⏬ تنزيل streaming_t2v.ckpt ...')
    !wget -q --show-progress -O {ckpt_path} https://huggingface.co/PAIR/StreamingT2V/resolve/main/streaming_t2v.ckpt
print('✅ Checkpoint جاهز:', ckpt_path, ckpt_path.stat().st_size, 'bytes')

## 4) إعداد ngrok والتوكنات

نقرأ التوكنات من Colab Secrets أوّلاً، ثم من متغيرات البيئة، ثم نطلبها تفاعليًا كحلّ أخير.

In [ ]:
import os, getpass

def _read_secret(name):
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name)
    except Exception:
        return None

ngrok_token = _read_secret('NGROK_AUTHTOKEN') or os.environ.get('NGROK_AUTHTOKEN')
if not ngrok_token:
    ngrok_token = getpass.getpass('NGROK_AUTHTOKEN (احصل عليه من dashboard.ngrok.com): ').strip()
os.environ['NGROK_AUTHTOKEN'] = ngrok_token

scout_token = _read_secret('SCOUT_API_TOKEN') or os.environ.get('SCOUT_API_TOKEN', '')
os.environ['SCOUT_API_TOKEN'] = (scout_token or '').strip()
if os.environ['SCOUT_API_TOKEN']:
    print('🔐 سيتم تفعيل الحماية بالتوكن.')
else:
    print('ℹ️  لم يتم ضبط SCOUT_API_TOKEN — السيرفر مفتوح بدون حماية.')

from pyngrok import ngrok, conf
conf.get_default().auth_token = ngrok_token
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)
print('✅ ngrok جاهز.')

## 5) كتابة كود السيرفر داخل Colab

نضع نفس محتوى ملف `colab/server.py` الموجود في الريبو حتى يعمل الـ Notebook منفردًا بدون استنساخ scout-app.

In [ ]:
import urllib.request, pathlib
server_path = pathlib.Path('/content/server.py')
SERVER_URL = 'https://raw.githubusercontent.com/toxichassan22/scout-app/main/colab/server.py'
try:
    urllib.request.urlretrieve(SERVER_URL, server_path)
    print('✅ تم تنزيل server.py من scout-app.')
except Exception as exc:
    raise SystemExit(f'تعذّر تنزيل server.py: {exc}\n'
                     'تأكد أنّ مستودع scout-app متاح للعموم أو ارفع الملف يدويًا إلى /content/server.py.')

## 6) تشغيل السيرفر وفتح نفق ngrok

هذه الخلية تظل قيد التشغيل ما دام الـ Notebook مفتوحًا. ستظهر الرسالة `Public URL: https://...ngrok-free.app` في الأسفل — انسخها وضعها في لوحة الأدمن.

In [ ]:
import os, sys, time, threading, subprocess
from pyngrok import ngrok

PORT = int(os.environ.get('STREAMING_PORT', '8000'))
log_path = '/content/server.log'
open(log_path, 'w').close()

server_proc = subprocess.Popen(
    [sys.executable, '/content/server.py'],
    stdout=open(log_path, 'ab'),
    stderr=subprocess.STDOUT,
    env={**os.environ},
)

# انتظر حتى يستجيب السيرفر محليًا
import urllib.request
for _ in range(60):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{PORT}/', timeout=1).read()
        break
    except Exception:
        time.sleep(1)
else:
    print('⚠️ السيرفر لم يستجب بعد 60 ثانية. آخر السجلات:')
    print(open(log_path).read()[-2000:])

tunnel = ngrok.connect(PORT, bind_tls=True)
print('=' * 60)
print(f'>>> Public URL: {tunnel.public_url}')
print('Health check:', f'{tunnel.public_url}/')
print('=' * 60)

def _tail_logs():
    with open(log_path, 'r') as fh:
        fh.seek(0, 2)
        while server_proc.poll() is None:
            line = fh.readline()
            if line:
                print(line, end='')
            else:
                time.sleep(0.5)

threading.Thread(target=_tail_logs, daemon=True).start()

try:
    server_proc.wait()
finally:
    ngrok.disconnect(tunnel.public_url)
    print('🛑 تم إيقاف نفق ngrok.')

## 7) اختبار سريع (اختياري)

شغّل هذه الخلية في تبويب آخر بعد ظهور الرابط العام للتأكد أنّ كل شيء يعمل.

In [ ]:
import os, time, requests
BASE_URL = 'https://REPLACE_WITH_NGROK_URL'  # ضع الرابط هنا
headers = {}
if os.environ.get('SCOUT_API_TOKEN'):
    headers['x-api-token'] = os.environ['SCOUT_API_TOKEN']
r = requests.post(f'{BASE_URL}/generate', json={
    'prompt': 'Cinematic shot of scouts collaborating on a digital project at night',
    'team_name': 'test',
    'num_frames': 32,
}, headers=headers, timeout=30)
print(r.status_code, r.json())
job_id = r.json()['job_id']
while True:
    s = requests.get(f'{BASE_URL}/status/{job_id}', headers=headers, timeout=15).json()
    print(s['status'], s.get('queue_position'))
    if s['status'] in ('done', 'failed'):
        print(s)
        break
    time.sleep(15)